In [ ]:
# ============================================================
# TASK 3 - GOOGLE PLAY STORE ANALYTICS
# ONE CELL: Python + HTML Dashboard
# ============================================================

import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime, time
from zoneinfo import ZoneInfo
import webbrowser
import json

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv("Play Store Data.csv")

print("Dataset loaded successfully!")
print("Original shape:", df.shape)


# ------------------------------------------------------------
# 2. CLEAN DATA
# ------------------------------------------------------------

# Clean Installs
df["Installs"] = (
    df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")

# Clean Reviews
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")

# Clean Last Updated
df["Last Updated"] = pd.to_datetime(
    df["Last Updated"],
    errors="coerce"
)

# Remove missing required values
task3 = df.dropna(
    subset=[
        "App",
        "Category",
        "Installs",
        "Reviews",
        "Last Updated"
    ]
).copy()


# ------------------------------------------------------------
# 3. TASK 3 FILTERS
# ------------------------------------------------------------

# Reviews should be more than 500
task3 = task3[task3["Reviews"] > 500]

# App name should NOT start with X, Y or Z
task3 = task3[
    ~task3["App"]
    .str.upper()
    .str.startswith(("X", "Y", "Z"))
]

# App name should NOT contain letter S
task3 = task3[
    ~task3["App"]
    .str.upper()
    .str.contains("S", regex=False)
]

# Keep the categories required for translation
task3 = task3[
    task3["Category"].isin(
        ["BEAUTY", "BUSINESS", "DATING"]
    )
].copy()


# ------------------------------------------------------------
# 4. TRANSLATE CATEGORIES
# ------------------------------------------------------------

category_translation = {
    "BEAUTY": "सौंदर्य",       # Hindi
    "BUSINESS": "வணிகம்",      # Tamil
    "DATING": "Partnersuche"   # German
}

task3["Display_Category"] = (
    task3["Category"]
    .map(category_translation)
)


# ------------------------------------------------------------
# 5. CREATE MONTHLY TIME SERIES
# ------------------------------------------------------------

task3["Month"] = (
    task3["Last Updated"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly = (
    task3
    .groupby(
        [
            "Category",
            "Display_Category",
            "Month"
        ]
    )["Installs"]
    .sum()
    .reset_index(name="Total_Installs")
)

monthly = monthly.sort_values(
    ["Category", "Month"]
)


# ------------------------------------------------------------
# 6. MONTH-OVER-MONTH GROWTH
# ------------------------------------------------------------

monthly["MoM_Growth"] = (
    monthly
    .groupby("Category")["Total_Installs"]
    .pct_change()
    * 100
)

# Significant growth = more than 20%
monthly["Significant_Growth"] = (
    monthly["MoM_Growth"] > 20
)


# ------------------------------------------------------------
# 7. CREATE PLOTLY GRAPH
# ------------------------------------------------------------

fig = go.Figure()

categories = monthly["Category"].unique()

for category in categories:

    category_data = monthly[
        monthly["Category"] == category
    ].sort_values("Month")

    display_category = category_data[
        "Display_Category"
    ].iloc[0]

    # Line
    fig.add_trace(
        go.Scatter(
            x=category_data["Month"],
            y=category_data["Total_Installs"],
            mode="lines+markers",
            name=display_category,
            customdata=category_data[
                ["MoM_Growth", "Significant_Growth"]
            ],
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Month: %{x|%b %Y}<br>"
                "Total Installs: %{y:,}<br>"
                "MoM Growth: %{customdata[0]:.2f}%"
                "<extra></extra>"
            )
        )
    )

    # --------------------------------------------------------
    # SHADE PERIODS WHERE GROWTH > 20%
    # --------------------------------------------------------

    for i in range(1, len(category_data)):

        row = category_data.iloc[i]
        previous = category_data.iloc[i - 1]

        if row["Significant_Growth"]:

            fig.add_trace(
                go.Scatter(
                    x=[
                        previous["Month"],
                        row["Month"],
                        row["Month"],
                        previous["Month"]
                    ],
                    y=[
                        0,
                        0,
                        row["Total_Installs"],
                        previous["Total_Installs"]
                    ],
                    fill="toself",
                    fillcolor="rgba(255,165,0,0.25)",
                    line=dict(width=0),
                    hoverinfo="skip",
                    showlegend=False
                )
            )


# ------------------------------------------------------------
# 8. GRAPH DESIGN
# ------------------------------------------------------------

fig.update_layout(
    title={
        "text": (
            "Task 3 - Monthly Install Trends by App Category"
        ),
        "x": 0.5
    },

    xaxis_title="Month",

    yaxis_title="Total Installs",

    hovermode="x unified",

    template="plotly_white",

    height=650,

    legend_title="App Category",

    margin=dict(
        l=70,
        r=40,
        t=90,
        b=70
    )
)


# ------------------------------------------------------------
# 9. CONVERT GRAPH TO HTML
# ------------------------------------------------------------

plot_html = fig.to_html(
    full_html=False,
    include_plotlyjs=True
)


# ------------------------------------------------------------
# 10. CREATE DASHBOARD HTML
# ------------------------------------------------------------

dashboard_html = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>
Google Play Store Analytics - Task 3
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    background-color: #f5f5f5;
    margin: 0;
    padding: 0;
}}

.header {{
    background-color: #222;
    color: white;
    padding: 25px;
    text-align: center;
}}

.container {{
    width: 95%;
    margin: 25px auto;
}}

.card {{
    background-color: white;
    padding: 25px;
    border-radius: 10px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.15);
}}

.info {{
    background-color: #eeeeee;
    padding: 15px;
    border-radius: 8px;
    margin-bottom: 20px;
}}

.warning {{
    text-align: center;
    padding: 60px;
    font-size: 22px;
    color: #555;
}}

</style>

</head>


<body>


<div class="header">

<h1>
Google Play Store Analytics
</h1>

<h2>
Monthly App Install Growth Analysis - Install Trend Analysis
</h2>

</div>


<div class="container">


<div id="dashboard">


<div class="card">


<div class="info">

<b>Task 3 Requirements:</b>

<br><br>

• Monthly total installs by app category
<br>
• Significant growth highlighted when MoM growth exceeds 20%
<br>
• Reviews greater than 500
<br>
• App name does not start with X, Y or Z
<br>
• App name does not contain letter S
<br>
• Beauty → सौंदर्य
<br>
• Business → வணிகம்
<br>
• Dating → Partnersuche
<br>
• Available: 6:00 PM - 9:00 PM IST

</div>


{plot_html}


</div>


</div>


<div id="outsideTime" class="warning">

<h2>
Monthly App Install Growth Analysis is currently unavailable
</h2>

<p>
Available: 6:00 PM - 9:00 PM IST
</p>

</div>


</div>


<script>

function checkTime() {{

    // Current time in India
    const now = new Date();

    const formatter = new Intl.DateTimeFormat(
        "en-US",
        {{
            timeZone: "Asia/Kolkata",
            hour: "2-digit",
            minute: "2-digit",
            hour12: false
        }}
    );

    const parts = formatter.formatToParts(now);

    let hour = parseInt(
        parts.find(
            p => p.type === "hour"
        ).value
    );

    let minute = parseInt(
        parts.find(
            p => p.type === "minute"
        ).value
    );


    const currentMinutes =
        hour * 60 + minute;


    const startMinutes =
        18 * 60;

    const endMinutes =
        21 * 60;


    const dashboard =
        document.getElementById(
            "dashboard"
        );

    const outsideTime =
        document.getElementById(
            "outsideTime"
        );


    if (
        currentMinutes >= startMinutes &&
        currentMinutes < endMinutes
    ) {{

        dashboard.style.display = "block";

        outsideTime.style.display = "none";

    }} else {{

        dashboard.style.display = "none";

        outsideTime.style.display = "block";

    }}

}}


// Check immediately
checkTime();


// Check every minute
setInterval(
    checkTime,
    60000
);

</script>


</body>

</html>
"""


# ------------------------------------------------------------
# 11. SAVE HTML FILE
# ------------------------------------------------------------

dashboard_file = Path(
    "dashboard_task3.html"
)

dashboard_file.write_text(
    dashboard_html,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 12. AUTOMATICALLY OPEN IN BROWSER
# ------------------------------------------------------------

webbrowser.open(
    dashboard_file.resolve().as_uri()
)


# ------------------------------------------------------------
# 13. ALSO SHOW GRAPH IN JUPYTER
# ------------------------------------------------------------

fig.show()


# ------------------------------------------------------------
# 14. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("==========================================")
print("       TASK 3 COMPLETED")
print("==========================================")
print()
print("Original rows:", len(df))
print("Filtered rows:", len(task3))
print("Monthly records:", len(monthly))
print()
print("Categories:")
print(monthly["Display_Category"].unique())
print()
print("Dashboard file:")
print(dashboard_file.resolve())
print()
print("Available: 6:00 PM - 9:00 PM IST")
print()
print("The dashboard has been opened automatically")
print("in your browser.")
print()
print("==========================================")

Dataset loaded successfully!
Original shape: (10841, 13)



       TASK 3 COMPLETED

Original rows: 10841
Filtered rows: 75
Monthly records: 25

Categories:
<StringArray>
['सौंदर्य', 'வணிகம்', 'Partnersuche']
Length: 3, dtype: str

Dashboard file:
C:\Users\Ranjita\Downloads\Google-Play-Store-Analytics-main (1)\Google-Play-Store-Analytics-main\dashboard_task3.html

Available: 6:00 PM - 9:00 PM IST

The dashboard has been opened automatically
in your browser.

